# Task 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [ ]:
d_model = 30
seq_len = 100
batch_size = 2

In [ ]:
# Слой эмбеддинга: num_embeddings – размер словаря, embedding_dim = d_model
emb_layer = nn.Embedding(num_embeddings=10000, embedding_dim=d_model)
tokens = torch.randint(0, 10000, (seq_len, batch_size))  # случайные токены
x = emb_layer(tokens)  # форма (seq_len, batch_size, d_model)

In [ ]:
tokens.shape

In [ ]:
tokens

In [ ]:
x.shape

In [ ]:
x[0]

In [ ]:
# Синусно-косинусное позиционное кодирование
pos = torch.arange(seq_len).unsqueeze(1)

In [ ]:
pos

In [ ]:
i = torch.arange(0, d_model, 2) # torch.arange(5) даёт [0,1,2,3,4]. 

In [ ]:
i

In [ ]:
angle_rates = 1 / torch.pow(10000, i /d_model) # Для каждой "пары" вычисляем
# множитель – это базовая частота синусно‑косинусной волны для компоненты с шагом 

In [ ]:
angle_rates

In [ ]:
# Инициализируем матрицу позиционных кодов нулями формы (5,64) - по одной строке 
# на каждую позицию и по одному столбцу на каждую компоненту вектора.
pos_enc = torch.zeros(seq_len, d_model)

In [ ]:
pos_enc
pos_enc[:, 0::2] = torch.sin(pos * angle_rates)   # чётные индексы
pos_enc[:, 1::2] = torch.cos(pos * angle_rates)   # нечётные

In [ ]:
pos_enc.shape

In [ ]:
# Визуализация
plt.figure(figsize=(12, 6))
plt.imshow(pos_enc, aspect='auto', cmap='viridis')
plt.xlabel('Размерность (d_model)')
plt.ylabel('Позиция в последовательности')
plt.title('Синусно-косинусное позиционное кодирование')
plt.colorbar(label='PE значение')
plt.tight_layout()
plt.show() 

In [ ]:
pos_enc = pos_enc.unsqueeze(1).repeat(1, batch_size, 1)  # (seq_len, batch_size, d_model) 
# При этом pos автоматически растягивается вдоль 64‑мерного вектора, а angle_rates - вдоль 5 позиций, и происходит покомпонентное умножение.
# unsqueeze(1) превращает (5,64) в (5,1,64).
# repeat(1,2,1) дублирует этот столбец дважды, получая тензор (5,2,64), одинаковый для всех 
# примеров в пакете.


In [ ]:
h = x + pos_enc  # результирующий тензор формы (seq_len, batch_size, d_model)

In [ ]:
h[:,1,:].shape

In [ ]:
print(h.shape)  # (5, 2, 64) 

# Task 2

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class FeedForward(nn.Module):
  # Ваш код здесь
  def __init__(self, d_model, dim_ff):
    super().__init__()
    self.linear1 = nn.Linear(d_model, dim_ff)
    self.relu = nn.ReLU()
    self.linear2 = nn.Linear(dim_ff, d_model)

  def forward(self, x):
    out = self.linear1(x)
    out = self.relu(out)
    out = self.linear2(out)
    return out



In [ ]:
# Проверим, что FeedForward не меняет форму тензора
d_model, dim_ff = 64, 256 # Ваш код здесь
ff_block = FeedForward(d_model, dim_ff) # Ваш код здесь
seq_len, batch_size = 5, 2# Ваш код здесь

In [ ]:


x = torch.randn(seq_len, batch_size, d_model)
y = ff_block(x) # Ваш код здесь
print(x.shape, '->', y.shape)

# Task 3

In [ ]:
class TransformerEncoderBlock(nn.Module):
  # Ваш код здесь
  def __init__(self, d_model, num_heads, dim_ff):
    super().__init__()
    self.self_attn = nn.MultiheadAttention(d_model, num_heads)
    self.norm1 = nn.LayerNorm(d_model)
    self.ff = FeedForward(d_model, dim_ff)
    self.norm2 = nn.LayerNorm(d_model)

  def forward(self, x):
    # x shape: (seq_len, batch_size, d_model)
    attn_out, _ = self.self_attn(x, x, x)     # Multi-Head Attention
    x = self.norm1(x + attn_out)              # Add & Norm
    ff_out = self.ff(x)                       # FeedForward
    x = self.norm2(x + ff_out)               # Add & Norm
    return x



In [ ]:
# Использование блока энкодера:
d_model, num_heads, dim_ff = 64, 8, 256 # Ваш код здесь
encoder_block = TransformerEncoderBlock(d_model, num_heads, dim_ff) # Ваш код здесь
x = torch.randn(10, 2, d_model)          # (seq_len=10, batch_size=2, d_model=64)
y = encoder_block(x)
print(y.shape)  # (10, 2, 64)

# P2

In [1]:
import torch
import torch.nn as nn

In [2]:
class MyMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model должно делиться на num_heads"
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        # Линейные слои для Q, K, V и выходной слой
        # Ваш код здесь
        self.WQ = nn.Linear(d_model, d_model, bias=False)
        self.WK = nn.Linear(d_model, d_model, bias=False)
        self.WV = nn.Linear(d_model, d_model, bias=False)

        self.WO = nn.Linear(d_model, d_model, bias=False) 

    def forward(self, query, key, value, mask=None):
        # query, key, value: (seq_len, batch, d_model)
        seq_len, batch, d_model = query.shape

        # Проекция Q, K, V
        # Ваш код здесь
        Q = self.WQ(query)  # (seq_len, batch, d_model)
        K = self.WK(key)
        V = self.WV(value)

        # Переставим размерности так, чтобы голова была отдельным измерением
        # Сначала делаем (batch, seq_len, d_model)
        # Ваш код здесь
        Q = Q.permute(1, 0, 2)
        K = K.permute(1, 0, 2)
        V = V.permute(1, 0, 2)


        # Затем разбиваем по головам и снова транспонируем:
        # (batch, num_heads, seq_len, head_dim)
        # Ваш код здесь
        Q = Q.view(batch, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch, seq_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        # Скалярное произведение и softmax
        # Ваш код здесь
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (batch, heads, seq, seq)

        if mask is not None:
            # Расширяем маску до (batch, heads, seq, seq) и обнуляем запрещённые связи
            mask = mask.unsqueeze(0).unsqueeze(1)  # (1,1,seq,seq)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attention = torch.softmax(scores, dim=-1)  # (batch, heads, seq, seq)

        # Взвешенное суммирование значений
        # Ваш код здесь
        out = torch.matmul(attention, V)  # (batch, heads, seq, head_dim)

        # Объединяем головы обратно: (batch, seq_len, d_model)
        # Ваш код здесь
        out = out.permute(0, 2, 1, 3).contiguous().view(batch, seq_len, d_model)

        # Финальный линейный слой
        # Ваш код здесь
        out = self.WO(out)  # (batch, seq_len, d_model)

        # Возвращаем в формат (seq_len, batch, d_model)
        # Ваш код здесь
        out = out.permute(1, 0, 2)

        return out

In [3]:
# Использование класса MyMultiHeadAttention:
d_model, num_heads = 64, 8
mha = MyMultiHeadAttention(d_model, num_heads)
x = torch.randn(10, 2, d_model)    # (seq_len=10, batch=2, d_model=64)
y = mha(x, x, x, mask=None)
print(y.shape)  # (10, 2, 64) 

torch.Size([10, 2, 64])
